# V7_C_N01 — Attendance and Dropout Early-Warning Support

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft v0.1. This notebook supports learning and review; it does not authorize an operational decision.

## Purpose and safeguarding boundary
Prioritize proportionate human review and learner support. The notebook must not label a child as a dropout, automate sanctions, or expose identities.

In [1]:
import numpy as np, pandas as pd
rng=np.random.default_rng(73)
n=360
df=pd.DataFrame({'learner_id':[f'L{i:04d}' for i in range(n)],'school':rng.choice([f'S{i:02d}' for i in range(12)],n),'grade':rng.integers(6,13,n),'attendance_30d':rng.uniform(.45,1,n),'attendance_change':rng.normal(-.02,.10,n),'assessment_change':rng.normal(0,.6,n),'distance_km':rng.gamma(2,2,n),'support_contact':rng.choice([0,1],n,p=[.72,.28])})
df.head()

## Data minimization
Only variables necessary for support triage are used. Protected characteristics are not needed to identify a learner; where lawfully available, they belong in fairness evaluation rather than casual display.

In [2]:
assert df.learner_id.is_unique
assert df.attendance_30d.between(0,1).all()
print('ROWS',len(df),'SCHOOLS',df.school.nunique())

ROWS 360 SCHOOLS 12


## Interpretable baseline
The score combines current absence, worsening attendance, assessment decline, and travel burden. It is deliberately transparent and must be locally reviewed.

In [3]:
def scale01(x):
 lo,hi=x.quantile(.01),x.quantile(.99)
 return ((x-lo)/(hi-lo)).clip(0,1)
df['absence']=1-df.attendance_30d
df['worsening_attendance']=scale01(-df.attendance_change)
df['worsening_assessment']=scale01(-df.assessment_change)
df['distance_burden']=scale01(df.distance_km)
df['support_priority']=.45*df.absence+.25*df.worsening_attendance+.15*df.worsening_assessment+.15*df.distance_burden
df.support_priority.describe().round(3)

## Abstention and reason codes
Borderline scores are not forced into a yes/no answer. Every referral carries an interpretable reason for human review.

In [4]:
df['disposition']=np.select([df.support_priority>=.52,df.support_priority<=.35],['REVIEW','ROUTINE'],default='ABSTAIN')
features=['absence','worsening_attendance','worsening_assessment','distance_burden']
df['reason_code']=df[features].idxmax(axis=1)
df.disposition.value_counts()

## Capacity-aware allocation
A support system must respect the number of cases schools can review. We select at most three per school; this is a workload rule, not a prediction threshold.

In [5]:
capacity=3
worklist=(df[df.disposition=='REVIEW'].sort_values(['school','support_priority'],ascending=[True,False]).groupby('school',group_keys=False).head(capacity))
assert worklist.groupby('school').size().le(capacity).all()
worklist[['learner_id','school','grade','support_priority','reason_code']].head().round(3)

## Distribution and fairness checks
Compare referral rates across grades and schools. Differences are prompts for investigation; they are not proof of discrimination or proof of fairness.

In [6]:
grade_rates=(df.assign(referred=df.learner_id.isin(worklist.learner_id)).groupby('grade').referred.mean())
print(grade_rates.round(3).to_string())
print('MAX_MIN_GAP',round(grade_rates.max()-grade_rates.min(),3))

grade
6     0.043
7     0.097
8     0.072
9     0.020
10    0.085
11    0.178
12    0.095
MAX_MIN_GAP 0.157


## Privacy-safe operational output
The analytical review product contains pseudonymous identifiers, reason codes, and status. Re-identification and contact occur only inside the authorized school workflow.

In [7]:
output=worklist[['learner_id','school','grade','support_priority','reason_code']].copy()
output['status']='PENDING SAFEGUARDING REVIEW'
assert output.learner_id.is_unique
output.head()

## Exercises
1. Replace the top-three rule with different capacities by school. 2. Add a data-latency abstention rule. 3. Explain why support_contact must not be interpreted as successful intervention. 4. Write a retention and deletion schedule.

## Solution guide
1. Join an authorized capacity table and use group-specific head(n) or optimization. 2. Abstain when the reporting delay exceeds the intervention horizon. 3. Contact is confounded by prior risk, access, and implementation; causal impact requires an evaluation design. 4. Specify purpose, lawful basis, access, active retention, archival/deletion trigger, audit log, and accountable custodian.

In [8]:
assert set(output.status)=={'PENDING SAFEGUARDING REVIEW'}
assert len(output)<=12*capacity
print('V7_C_N01_COMPLETE_EXECUTION_PASS')

V7_C_N01_COMPLETE_EXECUTION_PASS
